In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
data_path = "TMDB_movie_dataset_v11.csv"

def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df

movie_data = load_dataset(data_path)
print(movie_data.shape)
print(movie_data.columns)

(1363921, 24)
Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords'],
      dtype='object')


In [36]:
cols = [
    "id", "title", "release_date", "original_language", "overview",
    "popularity", "tagline", "genres", "keywords", "homepage"
    ]

use_cols = movie_data[cols].copy()
use_cols.isnull().sum()



id                         0
title                     18
release_date          287904
original_language          0
overview              308415
popularity                 0
tagline              1173993
genres                590449
keywords             1021524
homepage             1223025
dtype: int64

In [37]:
str_cols = ["title", "overview", "tagline", "genres", "keywords"]
for c in str_cols:
    use_cols[c] = use_cols[c].fillna("")

use_cols["release_date"] = pd.to_datetime(use_cols["release_date"], errors="coerce")
use_cols["release_year"] = use_cols["release_date"].dt.year
use_cols["release_year"] = use_cols["release_year"].fillna(0).astype(int)

use_cols["homepage"] = use_cols["homepage"].fillna("")

use_cols.isnull().sum()

id                        0
title                     0
release_date         287904
original_language         0
overview                  0
popularity                0
tagline                   0
genres                    0
keywords                  0
homepage                  0
release_year              0
dtype: int64

In [38]:
use_cols.dtypes

id                            int64
title                        object
release_date         datetime64[ns]
original_language            object
overview                     object
popularity                  float64
tagline                      object
genres                       object
keywords                     object
homepage                     object
release_year                  int64
dtype: object

In [39]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

use_cols["title_clean"] = use_cols["title"].apply(clean_text)
use_cols["overview_clean"] = use_cols["overview"].apply(clean_text)
use_cols["tagline_clean"] = use_cols["tagline"].apply(clean_text)
# use_cols["genres_clean"] = use_cols["genres"].apply(clean_text)
# use_cols["keywords_clean"] = use_cols["keywords"].apply(clean_text)

use_cols.head(3)

,id,title,release_date,original_language,overview,popularity,tagline,genres,keywords,homepage,release_year,title_clean,overview_clean,tagline_clean
0,27205,Inception,2010-07-15,en,"Cobb, a skilled thief who commits corporate es...",83.952,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","rescue, mission, dream, airplane, paris, franc...",https://www.warnerbros.com/movies/inception,2010,inception,cobb a skilled thief who commits corporate esp...,your mind is the scene of the crime
1,157336,Interstellar,2014-11-05,en,The adventures of a group of explorers who mak...,140.241,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","rescue, future, spacecraft, race against time,...",http://www.interstellarmovie.net/,2014,interstellar,the adventures of a group of explorers who mak...,mankind was born on earth it was never meant t...
2,155,The Dark Knight,2008-07-16,en,Batman raises the stakes in his war on crime. ...,130.643,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","joker, sadism, chaos, secret identity, crime f...",https://www.warnerbros.com/movies/dark-knight/,2008,the dark knight,batman raises the stakes in his war on crime w...,welcome to a world without rules


In [40]:
def split_genres(genre_str):
    if not genre_str:
        return []
    return [g.strip().lower() for g in genre_str.split(",")]

def split_keywords(keyword_str):
    if not keyword_str:
        return []
    return [k.strip().lower() for k in keyword_str.split(",")]

use_cols["genre_list"] = use_cols["genres"].apply(split_genres)
use_cols["keyword_list"] = use_cols["keywords"].apply(split_keywords)

use_cols[["genres", "genre_list", "keywords", "keyword_list"]].head(3)

,genres,genre_list,keywords,keyword_list
0,"Action, Science Fiction, Adventure","[action, science fiction, adventure]","rescue, mission, dream, airplane, paris, franc...","[rescue, mission, dream, airplane, paris, fran..."
1,"Adventure, Drama, Science Fiction","[adventure, drama, science fiction]","rescue, future, spacecraft, race against time,...","[rescue, future, spacecraft, race against time..."
2,"Drama, Action, Crime, Thriller","[drama, action, crime, thriller]","joker, sadism, chaos, secret identity, crime f...","[joker, sadism, chaos, secret identity, crime ..."


In [41]:
use_cols["genre_text"] = use_cols["genre_list"].apply(lambda xs: " ".join(xs))
use_cols["keyword_text"] = use_cols["keyword_list"].apply(lambda xs: " ".join(xs))
use_cols[["genre_text", "keyword_text"]].head(3)

,genre_text,keyword_text
0,action science fiction adventure,rescue mission dream airplane paris france vir...
1,adventure drama science fiction,rescue future spacecraft race against time art...
2,drama action crime thriller,joker sadism chaos secret identity crime fight...


In [42]:
# for vectorization
def build_combined_text(row):
    
    return (
        row["title_clean"] + " " +
        row["title_clean"] + " " +
        row["overview_clean"] + " " +
        row["tagline_clean"] + " " +
        clean_text(row["genre_text"]) + " " +
        clean_text(row["keyword_text"])
    ).strip()

use_cols["combined_text"] = use_cols.apply(build_combined_text, axis=1)

use_cols[["title", "combined_text"]].head(2)

,title,combined_text
0,Inception,inception inception cobb a skilled thief who c...
1,Interstellar,interstellar interstellar the adventures of a ...


In [43]:
# fitted vectorizer to transform user input
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2)
)
# for all movies
movie_vectors = tfidf.fit_transform(use_cols["combined_text"])

movie_vectors.shape

(1363921, 50000)

In [44]:
# user input to TF_IDF vector
def vectorize_user_text(user_text):
    user_text = clean_text(user_text)
    return tfidf.transform([user_text])

# top-k candidates retrieval
def topk_candidates(user_text, K=200):
    user_text = clean_text(user_text)

    if user_text == "":
        return [], np.array([])
    
    user_vector = tfidf.transform([user_text])
    similarities = cosine_similarity(user_vector, movie_vectors).ravel()

    # top-K indices
    K = min(K, len(similarities))
    top_idx = np.argpartition(-similarities, range(K))[:K]
    top_idx = top_idx[np.argsort(-similarities[top_idx])]

    return top_idx.tolist(), similarities[top_idx]

user_text = "mafia comedy"
top_idx, top_scores = topk_candidates(user_text, K=10)

use_cols.loc[top_idx, ["title", "release_year", "genres", "popularity", "original_language"]].assign(tfidf_score=top_scores)

,title,release_year,genres,popularity,original_language,tfidf_score
368446,One Against the Mafia,0,,0.0000,ru,0.918416
1322694,Mafia skrab,2016,,0.6000,ar,0.918416
1353828,Mafia 2,2021,,0.6000,hi,0.918416
1353830,Mafia,2020,,0.6000,hi,0.918416
203290,Mafia Mexicana,1992,Action,1.1650,es,0.885654
1175116,SWITCHBLADEE MAFIA,2022,Thriller,0.6000,pt,0.882312
711754,Mafia,1964,Documentary,0.0143,de,0.879785
644132,Kebad Mafia,2025,Thriller,0.0214,am,0.865515
809234,The Last Mafia,0,"Action, Drama",0.1429,hi,0.806303
1146559,Mafia,0,"Action, Crime",0.6370,kn,0.779335


In [45]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer("all-MiniLM-L6-v2")

In [46]:
def normalize01(x):
    x = np.array(x, dtype=float)
    if len(x) == 0:
        return x
    mn, mx = x.min(), x.max()
    return (x - mn) / (mx - mn + 1e-9)

def genre_match_score(movie_genres_list, selected_genres):
    if not selected_genres:
        return 0.0
    mg = set([g.strip().lower() for g in movie_genres_list])
    sg = [g.strip().lower() for g in selected_genres if str(g).strip()]
    if not sg:
        return 0.0
    hits = sum(1 for g in sg if g in mg)
    return hits / len(sg)

def recommend_movies(user_text, selected_genres=None, top_n=10, K=200,
                    use_genre_filter=False,
                    w_sbert=0.65, w_tfidf=0.25, w_genre=0.10, w_pop=0.03):
    selected_genres = selected_genres or []
    user_text_clean = clean_text(user_text or "")

    # 1) Candidate retrieval using Step 5 function
    cand_idx, cand_tfidf_scores = topk_candidates(user_text_clean, K=K)

    # If user_text empty, build a query from genres for retrieval
    if (not user_text_clean) and selected_genres:
        genre_query = " ".join([g.strip().lower() for g in selected_genres])
        cand_idx, cand_tfidf_scores = topk_candidates(genre_query, K=K)

    # If still no signal -> fallback (popularity)
    if len(cand_idx) == 0:
        top = use_cols.sort_values("popularity", ascending=False).head(top_n)
        return top[["title", "release_year", "overview", "genres", "popularity", "original_language"]]

    # 2) Optional hard genre filter on candidates
    if use_genre_filter and selected_genres:
        mask = []
        sg = set([g.strip().lower() for g in selected_genres])
        for i in cand_idx:
            gl = use_cols.iloc[i]["genre_list"]
            mask.append(any(g in sg for g in gl))
        mask = np.array(mask, dtype=bool)

        cand_idx = list(np.array(cand_idx)[mask])
        cand_tfidf_scores = np.array(cand_tfidf_scores)[mask]

        # if filter becomes too strict, fallback to no filter
        if len(cand_idx) == 0:
            cand_idx, cand_tfidf_scores = topk_candidates(user_text_clean, K=K)

    # 3) SBERT reranking on candidates ONLY
    # If user_text is empty, rerank using genre query
    sbert_query = user_text_clean if user_text_clean else " ".join([g.strip().lower() for g in selected_genres])

    query_emb = sbert.encode([sbert_query], normalize_embeddings=True)

    cand_texts = use_cols.iloc[cand_idx]["combined_text"].tolist()
    cand_embs = sbert.encode(cand_texts, normalize_embeddings=True)

    sbert_scores = cos_sim(query_emb, cand_embs).ravel()  # ~0..1

    # 4) Genre score + popularity score (for candidates)
    genre_scores = np.array([
        genre_match_score(use_cols.iloc[i]["genre_list"], selected_genres)
        for i in cand_idx
    ], dtype=float)

    pop_scores = use_cols.iloc[cand_idx]["popularity"].to_numpy(dtype=float)
    pop_scores = normalize01(pop_scores)

    # 5) Normalize TF-IDF candidate scores (they're already 0..1-ish, but normalize for stable mixing)
    tfidf_scores = normalize01(cand_tfidf_scores)

    # 6) Final combined score
    final_scores = (
        w_sbert * sbert_scores +
        w_tfidf * tfidf_scores +
        w_genre * genre_scores +
        w_pop * pop_scores
    )

    # 7) Rank and return Top-N
    order = np.argsort(-final_scores)
    pick_idx = np.array(cand_idx)[order]

    out = use_cols.iloc[pick_idx][["title", "release_year", "overview", "genres", "popularity", "original_language", "homepage"]].copy()
    out["similarity_percent"] = (final_scores[order] * 100).round(2)

    out = out[out["overview"].str.strip() != ""]
    out = out.drop_duplicates(subset=["title", "release_year", "original_language"])
    
    out = out.head(top_n)
    return out.reset_index(drop=True)

In [47]:
# ---- quick test ----
user_text = "sad ending"
selected_genres = ["Romance"]
recommend_movies(user_text, selected_genres, top_n=10, K=200, use_genre_filter=False)

,title,release_year,overview,genres,popularity,original_language,homepage,similarity_percent
0,Sad Man,2015,A sad story about a a father's wrong choice th...,"Romance, Action",0.6000,ko,,54.46
1,Sad Scene,2015,A sad breakup between a sound staff called boo...,Romance,0.6000,ko,,50.15
2,Happy Ending,2015,"When the son is sad, the daddy makes the happy...",Drama,0.6000,sv,,47.23
3,Sad Day,2025,Sad Day by FKA Twigs.,,1.4000,en,,46.77
4,"I Was Happy, I Was Sad",2020,A description through my sad times and the new...,Documentary,0.6000,th,,44.94
5,Happy Ending,2023,In a world where success and betrayal are alwa...,Romance,0.7080,or,,44.44
6,Rurouni Kenshin Memorial Ending,2007,The last scene of the 10 Years Special DVD Box...,"Animation, Romance",1.8700,ja,,44.08
7,Happy Ending,0,Upcoming Film by Ullash,,0.6000,en,,43.20
8,The Ending,0,After an adventure grand enough for the movies...,,0.0000,en,,42.55
9,Happy Ending,2025,Details here,Drama,0.0929,tl,,42.25
